## Setup

- **API key**: The first code cell loads your OpenRouter API key from a `.env` file. [OpenRouter](https://openrouter.ai) is a service that gives you one place to call many AI models. An API key is a secret code that identifies you to the service. The `.env` values replace any existing environment variables with the same name. If no key is found, the cell asks you to type it in. Never commit your key to Git.
- **Model**: The `MODEL` variable in `.env` chooses the model. If it is not set, the notebook uses `openai/gpt-6-luna`. Model names are listed at [openrouter.ai/models](https://openrouter.ai/models). They can change, so check there if the default stops working.
- **Warnings**: You may see a `UserWarning` about Pydantic and Python 3.14 in cell 3's output. It is harmless and can be ignored.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openai/gpt-6-luna")  # swap on openrouter.ai/models if needed

# Prompt Pipelining Overview

Prompt pipelining means building a prompt from smaller pieces and joining them together, rather than writing one long block of text. The pieces used in this notebook are:

- 📄 **Templates**: Text with `{placeholders}` (gaps) that are filled in when the prompt runs.
- 🧱 **Modularity**: Pieces are reusable, like LEGO bricks.
- 👓 **Readability**: Short pieces are easier to read and check than one long prompt.
- 🔗 **Sequential flow**: You join templates or strings in order with `+`.
- 💬 **Chat-friendly**: You can build a conversation by stacking messages.

For prompts that include examples, see notebooks 12a and 12b (few-shot prompting).

Prompt pipelining keeps prompts organised as they grow, so each piece can be changed without rewriting the whole prompt.

In [2]:
from langchain_core.prompts import PromptTemplate

prompt = (
    PromptTemplate.from_template("I'm heading to {destination}. ")
    + "Recommend a great {activity} spot!")

prompt

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


PromptTemplate(input_variables=['activity', 'destination'], input_types={}, partial_variables={}, template="I'm heading to {destination}. Recommend a great {activity} spot!")

In [3]:
prompt = prompt + "\n\nAlso, any local delicacies I should try?"
prompt

PromptTemplate(input_variables=['activity', 'destination'], input_types={}, partial_variables={}, template="I'm heading to {destination}. Recommend a great {activity} spot!\n\nAlso, any local delicacies I should try?")

In [4]:
print(prompt.template)

I'm heading to {destination}. Recommend a great {activity} spot!

Also, any local delicacies I should try?


In [5]:
prompt.format(destination="Dublin", activity="dining")

"I'm heading to Dublin. Recommend a great dining spot!\n\nAlso, any local delicacies I should try?"

### Pipelining vs. multi-input prompts

Cell 3's template already has two inputs, `{destination}` and `{activity}`, so the difference is not how many inputs a prompt takes. It is how the prompt is built:

- 🚂 **Building**: Pipelining adds separate pieces one at a time with `+`. Each piece can be written and tested on its own.
- ♻️ **Reuse**: A piece such as `"Also, any local delicacies I should try?"` can be added to other prompts without rewriting them.
- 🎼 **Analogy**: Pipelining is like building a melody note by note. A multi-input prompt is more like a chord: all the parts are written as one block and played together.

## Using the prompt in a chain

A **chain** is a series of steps where each step's output feeds into the next. In LangChain (the library used in this notebook), the `|` symbol joins the steps. It is called the LCEL pipe, short for LangChain Expression Language. In `prompt | llm | output_parser`:

1. **`prompt`** fills in the placeholders.
2. **`llm`** (large language model, the AI that writes the reply) reads the filled-in prompt and generates text. Its `temperature` controls randomness. Lower values give more predictable replies, and higher values give more varied ones. Cell 10 uses 0.75.
3. **`output_parser`** (`StrOutputParser`) turns the model's reply into plain text.

`.stream()` prints the reply as it is generated, rather than all at once.

Every run gives different wording, even for the same question. Compare cells 10, 13, 15, 16 and 17: the saved outputs are examples, not fixed answers. Cell 13 also shows that adding the greeting question changes the reply.

In [6]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.output_parsers import StrOutputParser

In [7]:
llm = ChatOpenRouter(model=MODEL, temperature=0.75)

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

for chunk in chain.stream({"destination":"Dublin", "activity":"dining"}):
  print(chunk, end="", flush=True)

For a memorable meal, try **The Winding Stair**: a warmly atmospheric spot overlooking the Liffey, known for seasonal Irish cooking and local ingredients. Book ahead, especially for dinner.

While you’re in Dublin, look out for **Dublin coddle** (a comforting sausage, bacon and potato stew), **boxty** (Irish potato pancakes), and fresh **oysters or seafood chowder**. For a more modern, casual local favorite, try a **spice bag**—crispy chicken, chips and peppers tossed with a spicy seasoning.

In [8]:
prompt = prompt + " How should I greet the locals in a friendly, informal manner?"


In [9]:
prompt.template

"I'm heading to {destination}. Recommend a great {activity} spot!\n\nAlso, any local delicacies I should try? How should I greet the locals in a friendly, informal manner?"

In [10]:
chain = prompt | llm | output_parser

for chunk in chain.stream({"destination":"Dublin", "activity":"dining"}):
  print(chunk, end="", flush=True)

For a memorable meal, try **The Winding Stair** on Ormond Quay: a cosy, book-lined spot serving seasonal Irish food, with views over the River Liffey. **Book ahead**, especially for dinner.

Things to try:
- **Dublin coddle** — a hearty stew, traditionally made with sausages, bacon, potatoes and onions.
- **Boxty** — an Irish potato pancake.
- **A spice bag** — a popular Dublin takeaway of crispy chicken, chips and spicy seasoning.
- **Soda bread** with butter, if it’s on the table.

For a friendly, informal greeting, **“Hiya!”** or **“How’s it going?”** works well. A relaxed “How are you?” is good too—no need to put on an accent.

## Example Use Case: A Travel Chatbot

`TravelChatbot` is a **class**, a blueprint that groups data and actions together. Each chatbot keeps its own `base_prompt`, so `append_to_prompt` changes the template for every later run.

What to notice in the outputs:

- **Reply printed twice (cell 15):** `print` shows the reply as it streams in. The function then returns the full text, and Jupyter displays that returned value, so the reply appears again.
- **Whole prompt answered each time (cells 16 and 17):** Every run sends the full template. The Dublin dinner answer appears again before the new greeting question is answered.
- **Hedged answer (cell 15):** The reply starts with "If you mean Dublin, Ireland". The prompt does not say which Dublin, so the model checks first. Specific prompts get more useful answers.

### Limitations and responsible use

- **No memory**: The bot does not remember earlier replies. Each run is one fresh message built from the template.
- **Check the facts**: The saved outputs name real Dublin places and dishes. The notebook has not checked them, so verify before relying on them. Model replies can be wrong or out of date.
- **Protect your key**: Never commit your `.env` file or API key to Git.

In [11]:
class TravelChatbot:
    def __init__(self, base_template):
        self.model = ChatOpenRouter(model=MODEL, temperature=0.75)
        self.base_prompt = PromptTemplate.from_template(base_template)

    def append_to_prompt(self, additional_text):
        self.base_prompt += additional_text

    def run_chain(self, destination, activity):
        output_parser = StrOutputParser()
        chain = self.base_prompt | self.model | output_parser
        text = ""
        for chunk in chain.stream({"destination":destination, "activity":activity}):
          print(chunk, end="", flush=True)
          text += chunk
        return text

# Usage
base_template = "I'm heading to {destination}. Recommend a great {activity} spot!"

chatbot = TravelChatbot(base_template)

# Basic prompt
chatbot.run_chain(destination="Dublin", activity="dining")

If you mean **Dublin, Ireland**, try **The Winding Stair**: a cozy, book-lined restaurant overlooking the River Liffey, with a strong menu of Irish ingredients and a relaxed feel. It’s a lovely choice for a special dinner without being overly formal—book ahead, especially for a window table.

'If you mean **Dublin, Ireland**, try **The Winding Stair**: a cozy, book-lined restaurant overlooking the River Liffey, with a strong menu of Irish ingredients and a relaxed feel. It’s a lovely choice for a special dinner without being overly formal—book ahead, especially for a window table.'

In [12]:
# Append more to the prompt and run again
chatbot.append_to_prompt("\n\nAlso, any local delicacies I should try?")

chatbot.run_chain(destination="Dublin", activity="dining")

For a memorable meal, try **The Winding Stair**: a relaxed, well-regarded restaurant overlooking the River Liffey, with seasonal Irish cooking and plenty of local ingredients. Book ahead, especially for dinner.

A few Dublin and Irish specialties to look for:

- **Dublin coddle** — a hearty stew traditionally made with sausages, bacon, potatoes and onions.
- **Irish stew** — lamb or mutton slowly cooked with potatoes and vegetables.
- **Boxty** — a potato pancake, often served with a savory filling.
- **Seafood and oysters** — Ireland’s coast supplies excellent seafood; ask what’s local and in season.
- **Soda bread and brown bread** — worth ordering alongside a meal, or with soup.

For something more casual and distinctly modern-Dublin, try a **spice bag** from a takeaway.

'For a memorable meal, try **The Winding Stair**: a relaxed, well-regarded restaurant overlooking the River Liffey, with seasonal Irish cooking and plenty of local ingredients. Book ahead, especially for dinner.\n\nA few Dublin and Irish specialties to look for:\n\n- **Dublin coddle** — a hearty stew traditionally made with sausages, bacon, potatoes and onions.\n- **Irish stew** — lamb or mutton slowly cooked with potatoes and vegetables.\n- **Boxty** — a potato pancake, often served with a savory filling.\n- **Seafood and oysters** — Ireland’s coast supplies excellent seafood; ask what’s local and in season.\n- **Soda bread and brown bread** — worth ordering alongside a meal, or with soup.\n\nFor something more casual and distinctly modern-Dublin, try a **spice bag** from a takeaway.'

In [13]:
chatbot.append_to_prompt(" How should I greet the locals in a friendly, informal, jolly colloquial manner?")

chatbot.run_chain(destination="Dublin", activity="dining")

For a memorable meal, try **The Winding Stair** on the Liffey: it serves seasonal Irish cooking in a cosy, book-lined setting. Book ahead, especially for dinner, and check the current menu and hours before you go.

A few Dublin bites to seek out:
- **Dublin coddle** — a comforting stew of sausages, bacon, potatoes and onions.
- **Boxty** — a potato pancake, often served with savoury fillings.
- **A spice bag** — a much-loved Dublin takeaway of crispy chicken, chips and spicy seasoning.
- **Irish seafood** — look for oysters or Dublin Bay prawns if they’re on the menu.

For a friendly, informal hello, try **“Howya?”** or **“How’s the craic?”** (“How are things?”). **“Grand”** is a handy, cheerful answer. “Top of the morning” is more of a cliché than a usual greeting.

'For a memorable meal, try **The Winding Stair** on the Liffey: it serves seasonal Irish cooking in a cosy, book-lined setting. Book ahead, especially for dinner, and check the current menu and hours before you go.\n\nA few Dublin bites to seek out:\n- **Dublin coddle** — a comforting stew of sausages, bacon, potatoes and onions.\n- **Boxty** — a potato pancake, often served with savoury fillings.\n- **A spice bag** — a much-loved Dublin takeaway of crispy chicken, chips and spicy seasoning.\n- **Irish seafood** — look for oysters or Dublin Bay prawns if they’re on the menu.\n\nFor a friendly, informal hello, try **“Howya?”** or **“How’s the craic?”** (“How are things?”). **“Grand”** is a handy, cheerful answer. “Top of the morning” is more of a cliché than a usual greeting.'

## Chat Prompt Pipeline

Chat models read a list of **messages**, not one block of text. Each message has a role:

- 🎬 **`SystemMessage`**: Sets the scene and the rules, such as "Welcome to the West of Ireland Chat!"
- 🙋 **`HumanMessage`**: What the user says.
- 🤖 **`AIMessage`**: What the model said earlier. Here it is written by hand in the code, not generated by the model. Writing earlier replies yourself is one way to steer what the model says next.

Chat prompt pipelining joins these messages with `+`. The result is a `ChatPromptTemplate`, a prompt template made of messages (see cells 19 and 20). The `{input}` placeholder is filled in at run time, as cell 21 shows.

Cell 21 uses `temperature=0.3`, lower than the 0.75 used earlier. Lower temperature makes replies more predictable, but it does not guarantee the same answer. The reply is still playful.

In [14]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_openrouter import ChatOpenRouter

# Setting the scene with a West of Ireland-themed system message
prompt = SystemMessage(content="Welcome to the West of Ireland Chat! 🇮🇪")

# Constructing a chat flow with Irish wit
new_prompt = (
    prompt
    + HumanMessage(content="How's the form?")
    + AIMessage(content="Ah sure, not too bad at all. Did you hear about the Galway rain?")
    + "{input}"
)

# Formatting the chat with the user's response
print(new_prompt.format_messages(input="No, what about it?"))

print(new_prompt)

[SystemMessage(content='Welcome to the West of Ireland Chat! 🇮🇪', additional_kwargs={}, response_metadata={}), HumanMessage(content="How's the form?", additional_kwargs={}, response_metadata={}), AIMessage(content='Ah sure, not too bad at all. Did you hear about the Galway rain?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='No, what about it?', additional_kwargs={}, response_metadata={})]
input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessage(content='Welcome to the West of Ireland Chat! 🇮🇪', additional_kwargs={}, response_metadata={}), HumanMessage(content="How's the form?", additional_kwargs={}, response_metadata={}), AIMessage(content='Ah sure, not too bad at all. Did you hear about the Galway rain?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={

In [15]:
new_prompt.messages

[SystemMessage(content='Welcome to the West of Ireland Chat! 🇮🇪', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="How's the form?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Ah sure, not too bad at all. Did you hear about the Galway rain?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]

In [16]:
model = ChatOpenRouter(model=MODEL, temperature=0.3)

output_parser = StrOutputParser()

chain = new_prompt | model | output_parser

# Running the chatbot to get the response (using invoke instead of stream)
response = chain.invoke({"input":"No, what about it?"})
print(response)

Ah, just the usual Galway forecast: rain, a bit more rain, then a suspicious patch of sunshine that lasts about three minutes! 😄
